In [1]:
import pandas as pd

train_data_file = "./zhengqi_train.txt"
test_data_file =  "./zhengqi_test.txt"

train_data = pd.read_csv(train_data_file, sep='\t', encoding='utf-8')
test_data = pd.read_csv(test_data_file, sep='\t', encoding='utf-8')

In [2]:
epsilon=1e-5

#组交叉特征，可以自行定义，如增加： x*x/y, log(x)/y 等等
func_dict = {
            'add': lambda x,y: x+y,
            'mins': lambda x,y: x-y,
            'div': lambda x,y: x/(y+epsilon),
            'multi': lambda x,y: x*y
            }

In [3]:
def auto_features_make(train_data,test_data,func_dict,col_list):
    train_data, test_data = train_data.copy(), test_data.copy()
    for col_i in col_list:
        for col_j in col_list:
            for func_name, func in func_dict.items():
                for data in [train_data,test_data]:
                    func_features = func(data[col_i],data[col_j])
                    col_func_features = '-'.join([col_i,func_name,col_j])
                    data[col_func_features] = func_features
    return train_data,test_data

In [4]:
train_data2, test_data2 = auto_features_make(train_data,test_data,func_dict,col_list=test_data.columns)

In [5]:
from sklearn.decomposition import PCA   #主成分分析法

#PCA方法降维
pca = PCA(n_components=500)
train_data2_pca = pca.fit_transform(train_data2.iloc[:,0:-1])
test_data2_pca = pca.transform(test_data2)
train_data2_pca = pd.DataFrame(train_data2_pca)
test_data2_pca = pd.DataFrame(test_data2_pca)
train_data2_pca['target'] = train_data2['target']

In [6]:
X_train2 = train_data2[test_data2.columns].values
y_train = train_data2['target']

In [7]:
# ls_validation i
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import numpy as np

# 5折交叉验证
Folds=5
# kf = KFold(len(X_train2), n_splits=Folds, random_state=2019, shuffle=True)

# kf = KFold(len(X_train2), n_splits=Folds, random_state=2019, shuffle=True)

kf = KFold(n_splits =Folds, random_state=100, shuffle=True)  
kf.get_n_splits(len(X_train2))



# 记录训练和预测MSE
MSE_DICT = {
    'train_mse':[],
    'test_mse':[]
}

# 线下训练预测
for i, (train_index, test_index) in enumerate(kf.split(X_train2)):
    # lgb树模型
    lgb_reg = lgb.LGBMRegressor(
        learning_rate=0.01,
        max_depth=-1,
        n_estimators=5000,
        boosting_type='gbdt',
        random_state=2019,
        objective='regression',
    )
   
    # 切分训练集和预测集
    X_train_KFold, X_test_KFold = X_train2[train_index], X_train2[test_index]
    y_train_KFold, y_test_KFold = y_train[train_index], y_train[test_index]
     # 训练模型
    lgb_reg.fit(
            X=X_train_KFold,y=y_train_KFold,
            eval_set=[(X_train_KFold, y_train_KFold),(X_test_KFold, y_test_KFold)],
            eval_names=['Train','Test'],
            early_stopping_rounds=100,
            eval_metric='MSE',
            verbose=50
        )


    # 训练集预测 测试集预测
    y_train_KFold_predict = lgb_reg.predict(X_train_KFold,num_iteration=lgb_reg.best_iteration_)
    y_test_KFold_predict = lgb_reg.predict(X_test_KFold,num_iteration=lgb_reg.best_iteration_) 
    
    print('第{}折 训练和预测 训练MSE 预测MSE'.format(i))
    train_mse = mean_squared_error(y_train_KFold_predict, y_train_KFold)
    print('------\n', '训练MSE\n', train_mse, '\n------')
    test_mse = mean_squared_error(y_test_KFold_predict, y_test_KFold)
    print('------\n', '预测MSE\n', test_mse, '\n------\n')
    
    MSE_DICT['train_mse'].append(train_mse)
    MSE_DICT['test_mse'].append(test_mse)
print('------\n', '训练MSE\n', MSE_DICT['train_mse'], '\n', np.mean(MSE_DICT['train_mse']), '\n------')
print('------\n', '预测MSE\n', MSE_DICT['test_mse'], '\n', np.mean(MSE_DICT['test_mse']), '\n------')


Training until validation scores don't improve for 100 rounds.
[50]	Train's l2: 0.419929	Train's l2: 0.419929	Test's l2: 0.420996	Test's l2: 0.420996
[100]	Train's l2: 0.200329	Train's l2: 0.200329	Test's l2: 0.233619	Test's l2: 0.233619
[150]	Train's l2: 0.109049	Train's l2: 0.109049	Test's l2: 0.161406	Test's l2: 0.161406
[200]	Train's l2: 0.0678946	Train's l2: 0.0678946	Test's l2: 0.133299	Test's l2: 0.133299
[250]	Train's l2: 0.0472336	Train's l2: 0.0472336	Test's l2: 0.122054	Test's l2: 0.122054
[300]	Train's l2: 0.0353781	Train's l2: 0.0353781	Test's l2: 0.116745	Test's l2: 0.116745
[350]	Train's l2: 0.027622	Train's l2: 0.027622	Test's l2: 0.11317	Test's l2: 0.11317
[400]	Train's l2: 0.0221534	Train's l2: 0.0221534	Test's l2: 0.111322	Test's l2: 0.111322
[450]	Train's l2: 0.0179972	Train's l2: 0.0179972	Test's l2: 0.110315	Test's l2: 0.110315
[500]	Train's l2: 0.0148009	Train's l2: 0.0148009	Test's l2: 0.109687	Test's l2: 0.109687
[550]	Train's l2: 0.0122895	Train's l2: 0.012289

KeyboardInterrupt: 